# 🎭 Парсер ТК РФ через Playwright

**Цель**: Извлечь все статьи Трудового кодекса РФ с сайта ConsultantPlus

**URL**: https://www.consultant.ru/document/cons_doc_LAW_34683/

**Метод**: Браузерная автоматизация (Playwright) для рендеринга JavaScript

---

## План:
1. ✅ Установить playwright и браузеры
2. ⏳ Загрузить список статей (537 ссылок)
3. ⏳ Написать функцию для парсинга одной статьи
4. ⏳ Протестировать на статье 80
5. ⏳ Масштабировать на все статьи
6. ⏳ Сохранить в JSON

## Шаг 1: Установка Playwright

Запускаем один раз для установки пакета и браузера Chromium

In [5]:
!pip install playwright
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 5.7 MB/s  0:00:00m ? eta -:--:--
  Attempting uninstall: greenlet
    Found existing installation: greenlet 3.1.1
    Uninstalling greenlet-3.1.1:
      Successfully uninstalled greenlet-3.1.1

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
173.9 MiB [                    ] 0% 0.0s173.9 MiB [                    ] 0% 350.5s173.9 MiB [                    ] 0% 229.7s173.9 MiB [                    ] 0% 185.6s173.9 MiB [                    ] 0% 131.7s173.9 MiB [                    ] 0% 111.2s173.9 MiB [                    ] 0% 100.0s173.9 MiB [                    ] 0% 90.8s173.9 MiB [                    ] 0% 85.1s173.9 MiB [                    ] 0% 78.9s173.9 MiB [                    ] 0% 75.1s173.9 MiB [                    ] 0% 71.4s173.9 MiB [                    ] 0% 67.9s173.9 MiB [                    ] 0% 64.7s173.9 MiB [                    ] 0% 63.2s173.9 MiB [        

## Шаг 2: Загрузка списка статей

Используем простой HTTP запрос для получения ссылок (главная страница не требует JS)

In [ ]:
import httpx
from bs4 import BeautifulSoup
import re

URL = "https://www.consultant.ru/document/cons_doc_LAW_34683/"


# Загружаем главную страницу
async def fetch_article_links():
    async with httpx.AsyncClient(
        timeout=30.0, trust_env=False, follow_redirects=True
    ) as client:
        response = await client.get(URL)
        response.raise_for_status()
        return response.text


html = await fetch_article_links()
soup = BeautifulSoup(html, "html.parser")

# Извлекаем ссылки на статьи
article_links = []
for link in soup.find_all("a", href=True):
    text = link.get_text(strip=True)
    if "Статья" in text and re.search(r"\d+", text):
        article_links.append(
            {"text": text, "href": "https://www.consultant.ru" + link["href"]}
        )

print(f"✅ Найдено статей: {len(article_links)}")
print("\n📋 Первые 5 статей:")
for i, item in enumerate(article_links[:5], 1):
    print(f"{i}. {item['text']}")

✅ Найдено статей: 537

📋 Первые 5 статей:
1. Статья 1. Цели и задачи трудового законодательства
2. Статья 2. Основные принципы правового регулирования трудовых отношений и иных непосредственно связанных с ними отношений
3. Статья 3. Запрещение дискриминации в сфере труда
4. Статья 4. Запрещение принудительного труда
5. Статья 5. Трудовое законодательство и иные акты, содержащие нормы трудового права


## Шаг 2.5: Парсинг оглавления (иерархия Часть → Раздел → Глава)

Извлекаем структуру ТК РФ с главной страницы, чтобы потом привязать каждую статью к её иерархии

In [14]:
# Функция для извлечения иерархии из оглавления
def parse_tk_rf_hierarchy(soup):
    """
    Парсит оглавление ТК РФ и создает маппинг: номер статьи → иерархия

    Returns:
        dict: {
            '1': {'part': 'Часть I', 'section': '...', 'chapter': '...'},
            '2': {...},
            ...
        }
    """
    hierarchy_map = {}

    # Текущий контекст
    current_part = None
    current_section = None
    current_chapter = None

    # Ищем все элементы в оглавлении
    # ConsultantPlus обычно использует структуру списков или div'ов

    # Вариант 1: Поиск в основном контейнере с содержанием
    content = soup.find("div", class_="document")
    if not content:
        content = soup

    # Ищем все ссылки и заголовки
    for elem in content.find_all(["a", "div", "p", "h2", "h3", "h4"]):
        text = elem.get_text(strip=True)

        # Проверяем на "Часть"
        part_match = re.match(r"Часть\s+([IVXLCDM]+)", text, re.IGNORECASE)
        if part_match:
            current_part = f"Часть {part_match.group(1)}"
            print(f"✓ Найдена {current_part}")
            continue

        # Проверяем на "Раздел"
        section_match = re.match(
            r"(Раздел\s+[IVXLCDM]+\.?\s*[^\n]*)", text, re.IGNORECASE
        )
        if section_match:
            current_section = section_match.group(1).strip()
            print(f"  ✓ Найден {current_section[:50]}...")
            continue

        # Проверяем на "Глава"
        chapter_match = re.match(r"(Глава\s+\d+\.?\s*[^\n]*)", text, re.IGNORECASE)
        if chapter_match:
            current_chapter = chapter_match.group(1).strip()
            print(f"    ✓ Найдена {current_chapter[:50]}...")
            continue

        # Проверяем на "Статья"
        article_match = re.search(r"Статья\s+(\d+)", text)
        if article_match:
            article_num = article_match.group(1)
            hierarchy_map[article_num] = {
                "part": current_part,
                "section": current_section,
                "chapter": current_chapter,
            }

    return hierarchy_map


print("✅ Функция parse_tk_rf_hierarchy готова")

✅ Функция parse_tk_rf_hierarchy готова


In [15]:
# Парсим иерархию из уже загруженной главной страницы
article_hierarchy = parse_tk_rf_hierarchy(soup)

print(f"\n{'=' * 80}")
print(f"📊 Извлечено иерархий для {len(article_hierarchy)} статей")
print(f"{'=' * 80}\n")

# Проверяем несколько примеров
test_articles = ["1", "2", "80", "100"]
for art_num in test_articles:
    if art_num in article_hierarchy:
        h = article_hierarchy[art_num]
        print(f"Статья {art_num}:")
        print(f"  Часть: {h['part'] or 'не найдено'}")
        print(f"  Раздел: {h['section'] or 'не найдено'}")
        print(f"  Глава: {h['chapter'] or 'не найдено'}")
        print()

✓ Найдена Часть I
✓ Найдена Часть I
  ✓ Найден Раздел I. Общие положения...
    ✓ Найдена Глава 1. Основные начала трудового законодательств...
    ✓ Найдена Глава 2. Трудовые отношения, стороны трудовых отно...
✓ Найдена Часть II
  ✓ Найден Раздел II. Социальное партнерство в сфере труда...
    ✓ Найдена Глава 3. Общие положения...
    ✓ Найдена Глава 4. Представители работников и работодателей ...
    ✓ Найдена Глава 5. Органы социального партнерства...
    ✓ Найдена Глава 6. Коллективные переговоры...
    ✓ Найдена Глава 7. Коллективные договоры и соглашения...
    ✓ Найдена Глава 8. Участие работников в управлении организац...
    ✓ Найдена Глава 9. Ответственность сторон социального партне...
✓ Найдена Часть III
  ✓ Найден Раздел III. Трудовой договор...
    ✓ Найдена Глава 10. Общие положения...
    ✓ Найдена Глава 11. Заключение трудового договора...
    ✓ Найдена Глава 12. Изменение трудового договора...
    ✓ Найдена Глава 13. Прекращение трудового договора...
    ✓ Найдена Гл

## Шаг 3: Функция для парсинга одной статьи через Playwright

Создаем async функцию, которая:
1. Открывает браузер
2. Загружает страницу статьи
3. Ждет рендеринга JavaScript
4. Извлекает текст

In [6]:
from playwright.async_api import async_playwright


async def fetch_article_with_playwright(url: str, article_title: str):
    """
    Загружает статью через Playwright и извлекает текст

    Args:
        url: URL статьи
        article_title: Название статьи (для логирования)

    Returns:
        dict: {'title': str, 'text': str, 'url': str}
    """
    async with async_playwright() as p:
        # Запускаем браузер в headless режиме
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            # Загружаем страницу
            await page.goto(url, wait_until="networkidle", timeout=30000)

            # Ждем появления контента (div.doc-style содержит текст статьи)
            await page.wait_for_selector("div.doc-style", timeout=10000)

            # Извлекаем заголовок
            title_elem = await page.query_selector("h1")
            title = await title_elem.inner_text() if title_elem else article_title

            # Извлекаем текст статьи из div.doc-style
            content_elem = await page.query_selector("div.doc-style")
            if content_elem:
                # Получаем весь текст из контейнера
                text = await content_elem.inner_text()
            else:
                text = "Текст не найден"

            await browser.close()

            return {"title": title.strip(), "text": text.strip(), "url": url}

        except Exception as e:
            await browser.close()
            print(f"❌ Ошибка при загрузке {article_title}: {e}")
            return None


print("✅ Функция fetch_article_with_playwright готова")

✅ Функция fetch_article_with_playwright готова


## Шаг 4: Тест на статье 80

Проверим, что функция работает корректно

In [7]:
# Найдем статью 80
article_80 = None
for item in article_links:
    if "Статья 80" in item["text"]:
        article_80 = item
        break

if article_80:
    print(f"🔍 Загружаем: {article_80['text']}")
    print(f"🔗 URL: {article_80['href']}\n")

    # Загружаем статью
    result = await fetch_article_with_playwright(article_80["href"], article_80["text"])

    if result:
        print("\n✅ Статья загружена!")
        print(f"\n📌 Заголовок: {result['title']}")
        print(f"\n📝 Длина текста: {len(result['text'])} символов")
        print("\n📄 Первые 500 символов текста:\n")
        print("=" * 80)
        print(result["text"][:500])
        print("=" * 80)
else:
    print("❌ Статья 80 не найдена")

🔍 Загружаем: Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)
🔗 URL: https://www.consultant.ru/document/cons_doc_LAW_34683/aed7d03df679e3376974dadd131b899dc6966650/


✅ Статья загружена!

📌 Заголовок: ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)

📝 Длина текста: 97 символов

📄 Первые 500 символов текста:

ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)

✅ Статья загружена!

📌 Заголовок: ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)

📝 Длина текста: 97 символов

📄 Первые 500 символов текста:

ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)


### Шаг 4.1: Отладка - исследуем структуру страницы

Playwright загрузил только заголовок. Давайте посмотрим, что происходит со страницей

In [ ]:
async def debug_page_structure(url: str):
    """Отладка - смотрим структуру страницы"""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)  # Видимый браузер
        page = await browser.new_page()

        # Загружаем страницу
        await page.goto(url, wait_until="networkidle", timeout=30000)

        # Ждем div.doc-style
        await page.wait_for_selector("div.doc-style", timeout=10000)

        # Дополнительная пауза для JS
        await page.wait_for_timeout(3000)

        # Проверяем разные селекторы
        print("🔍 Проверяем селекторы:\n")

        selectors = [
            "div.doc-style",
            "div.document-text",
            "div.para",
            "p",
            "div.doc-style p",
            "div.doc-style div",
        ]

        for selector in selectors:
            elements = await page.query_selector_all(selector)
            print(f"{selector}: {len(elements)} элементов")

            if elements:
                first_elem = elements[0]
                text = await first_elem.inner_text()
                print(f"  Первый элемент ({len(text)} символов): {text[:100]}...")
                print()

        # Получаем весь HTML div.doc-style
        doc_style = await page.query_selector("div.doc-style")
        if doc_style:
            html_content = await doc_style.inner_html()
            print(f"\n📄 HTML внутри div.doc-style ({len(html_content)} символов):")
            print("=" * 80)
            print(html_content[:1000])
            print("=" * 80)

        # Держим браузер открытым для визуального осмотра
        print("\n⏸️  Браузер открыт. Осмотрите страницу вручную.")
        print("Нажмите Enter для закрытия...")
        input()

        await browser.close()


# Запускаем отладку
if article_80:
    await debug_page_structure(article_80["href"])

🔍 Проверяем селекторы:

div.doc-style: 1 элементов
  Первый элемент (97 символов): ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)...

div.document-text: 0 элементов
div.para: 0 элементов
p: 24 элементов
  Первый элемент (97 символов): ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)...

div.doc-style p: 1 элементов
  Первый элемент (97 символов): ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)...

div.doc-style div: 4 элементов
  Первый элемент (0 символов): ...


📄 HTML внутри div.doc-style (394 символов):
<h1><p><div class="info-link"><div class="info-link__button info-link__button_header"></div><div class="info-link__window info-link__window_hidden info-link__window_header"><button class="info-link__close"></button><div class="info-link__content"></div></div></div><a id="dst100579"></a>ТК РФ Статья 80. Расторжение трудового договора по

### Шаг 4.5: Альтернативная стратегия - ждем параграфы

Попробуем ждать конкретные элементы текста внутри контейнера

In [ ]:
import asyncio


async def fetch_article_v3(url: str, article_title: str):
    """
    Загружает статью - v3 с ожиданием параграфов
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            print(f"⏳ Загружаем: {article_title[:50]}...")

            # Загружаем страницу
            await page.goto(url, timeout=60000)

            # Ждем контейнер
            await page.wait_for_selector("div.document-page__content", timeout=20000)
            print("✓ Контейнер найден")

            # Стратегия: ждем появления параграфов внутри контейнера
            print("⏳ Ждем загрузки параграфов...")

            # Ждем, пока появятся параграфы с текстом (не пустые)
            await page.wait_for_function(
                """
                () => {
                    const container = document.querySelector('div.document-page__content');
                    if (!container) return false;
                    
                    const paragraphs = container.querySelectorAll('p');
                    let textLength = 0;
                    paragraphs.forEach(p => textLength += p.innerText.length);
                    
                    // Ждем, пока текст в параграфах превысит 300 символов
                    return textLength > 300;
                }
            """,
                timeout=60000,
            )

            print("✓ Параграфы загружены")

            # Небольшая пауза для стабилизации
            await asyncio.sleep(2)

            # Извлекаем заголовок
            title_elem = await page.query_selector("h1")
            title = await title_elem.inner_text() if title_elem else article_title

            # Извлекаем текст
            content_elem = await page.query_selector("div.document-page__content")
            if content_elem:
                text = await content_elem.inner_text()
            else:
                text = "Текст не найден"

            await browser.close()

            return {
                "title": title.strip(),
                "text": text.strip(),
                "url": url,
                "text_length": len(text.strip()),
            }

        except Exception as e:
            await browser.close()
            print(f"❌ Ошибка: {e}")
            return None


print("✅ Функция fetch_article_v3 готова")

✅ Функция fetch_article_v3 готова


### Шаг 4.6: Обновленная функция fetch_article_v4

Добавляем извлечение всех новых полей:
- part, section, chapter (из навигации или оглавления)
- status (проверяем текст на "утратила силу")
- source, source_url, fetched_at
- text_length (автоматически)

In [ ]:
async def parse_articles(base_url: str, articles: list[dict]) -> list[dict]:
    """
    Парсит статьи Трудового Кодекса РФ.

    Args:
        base_url: Базовый URL сайта Консультант Плюс
        articles: Список словарей со статьями (должны содержать 'url', 'title', 'number')

    Returns:
        Список статей с полным текстом
    """
    results = []

    async with async_playwright() as p:
        # Отключаем изображения и мультимедиа для ускорения
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--disable-dev-shm-usage",
            ],
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            viewport={"width": 1920, "height": 1080},
        )

        # Блокируем ненужные ресурсы
        await context.route(
            "**/*.{png,jpg,jpeg,gif,svg,css,woff,woff2}",
            handler=lambda route: route.abort(),
        )

        page = await context.new_page()

        for idx, article in enumerate(articles, 1):
            url = article.get("url", "")
            article_title = article.get("title", "")
            article_number = article.get("number", "")

            if not url:
                print(f"❌ Пропускаем {article_number}: нет URL")
                continue

            print(f"⏳ Загружаем: {article_title[:50]}...")

            # Загружаем страницу
            await page.goto(url, timeout=60000)

            # Ждем контейнер
            await page.wait_for_selector("div.document-page__content", timeout=20000)

            # Ждем появления любого текста в контейнере (для коротких статей типа "утратила силу")
            try:
                await page.wait_for_function(
                    """
                    () => {
                        const container = document.querySelector('div.document-page__content');
                        if (!container) return false;
                        // Проверяем весь текст в контейнере, не только в параграфах
                        const text = container.innerText || container.textContent || '';
                        return text.trim().length > 5;
                    }
                """,
                    timeout=10000,
                )
            except Exception:
                # Если не дождались, не страшно - попробуем извлечь что есть
                print("  ⚠️ Таймаут ожидания текста, пробуем извлечь...")

            await asyncio.sleep(1)

            # === ИЗВЛЕКАЕМ ЗАГОЛОВОК ===
            title_elem = await page.query_selector("h1")
            title = await title_elem.inner_text() if title_elem else article_title
            title = title.strip()

            # === ИЗВЛЕКАЕМ ТЕКСТ ===
            content_elem = await page.query_selector("div.document-page__content")
            if content_elem:
                text = await content_elem.inner_text()
                text = text.strip()
            else:
                text = "Текст не найден"

            # === ИЗВЛЕКАЕМ ПРИМЕЧАНИЕ ===
            footnote = ""
            footnote_elem = await page.query_selector("div.footnote")
            if footnote_elem:
                footnote = await footnote_elem.inner_text()
                footnote = footnote.strip()

            # === ОБРАБОТКА УТРАТИВШИХ СИЛУ ===
            if "утратил силу" in text.lower() or "утратила силу" in text.lower():
                print(f"  📜 Статья {article_number} утратила силу")
                status = "abolished"
            elif "приостановлен" in text.lower():
                status = "suspended"
            else:
                status = "active"

            results.append(
                {
                    "number": article_number,
                    "title": title,
                    "content": text,
                    "footnote": footnote,
                    "status": status,
                    "chapter": article.get("chapter", ""),
                    "url": url,
                    "text_length": len(text),
                }
            )

            print(f"  ✅ {article_number}: {len(text)} символов")

        await browser.close()

    return results

✅ Функция fetch_article_v4 готова (обновлена для коротких статей)


In [ ]:
from datetime import datetime
import re


async def fetch_article_v4(url: str, article_title: str, hierarchy_map: dict):
    """
    Загружает статью с полными метаданными (v4)

    Args:
        url: URL статьи
        article_title: Заголовок статьи (например, "Статья 80. Расторжение...")
        hierarchy_map: Словарь с иерархией {номер: {part, section, chapter}}

    Returns:
        dict с полями:
            - title, text, text_length, url
            - status: 'active' | 'abolished' | 'suspended'
            - part, section, chapter: иерархия из маппинга
            - source, source_url, fetched_at
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            print(f"⏳ Загружаем: {article_title[:50]}...")

            # Загружаем страницу
            await page.goto(url, timeout=60000)

            # Ждем контейнер
            await page.wait_for_selector("div.document-page__content", timeout=20000)
            print("✓ Контейнер найден")

            # Ждем появления текста (адаптировано для коротких статей)
            try:
                await page.wait_for_function(
                    """
                    () => {
                        const container = document.querySelector('div.document-page__content');
                        if (!container) return false;
                        const text = container.innerText || container.textContent || '';
                        return text.trim().length > 50;
                    }
                """,
                    timeout=15000,
                )
                print("✓ Текст загружен")
            except Exception:
                print("  ⚠️ Таймаут ожидания текста, продолжаем...")

            # Небольшая пауза для стабилизации
            await asyncio.sleep(1)

            # === ИЗВЛЕКАЕМ ЗАГОЛОВОК ===
            title_elem = await page.query_selector("h1")
            title = await title_elem.inner_text() if title_elem else article_title
            title = title.strip()

            # === ИЗВЛЕКАЕМ ТЕКСТ ===
            content_elem = await page.query_selector("div.document-page__content")
            if content_elem:
                text = await content_elem.inner_text()
                text = text.strip()
            else:
                text = "Текст не найден"

            # === ОПРЕДЕЛЯЕМ СТАТУС ===
            text_lower = text.lower()
            if "утратил силу" in text_lower or "утратила силу" in text_lower:
                status = "abolished"
            elif "приостановлен" in text_lower or "приостановлена" in text_lower:
                status = "suspended"
            else:
                status = "active"

            # === ИЗВЛЕКАЕМ НОМЕР СТАТЬИ ===
            article_num_match = re.search(r"Статья\s+(\d+)", article_title)
            article_number = article_num_match.group(1) if article_num_match else None

            # === ПОЛУЧАЕМ ИЕРАРХИЮ ИЗ МАППИНГА ===
            hierarchy = hierarchy_map.get(article_number, {}) if article_number else {}
            part = hierarchy.get("part")
            section = hierarchy.get("section")
            chapter = hierarchy.get("chapter")

            await browser.close()

            return {
                "title": title,
                "text": text,
                "text_length": len(text),
                "status": status,
                "part": part,
                "section": section,
                "chapter": chapter,
                "source": "ConsultantPlus",
                "source_url": url,
                "fetched_at": datetime.now().isoformat(),
            }

        except Exception as e:
            await browser.close()
            print(f"❌ Ошибка: {e}")
            return None


print("✅ Функция fetch_article_v4 готова")

In [23]:
# Тестируем v4 с иерархией на статье 80
if article_80:
    print("🚀 Тест v4: с извлечением иерархии из маппинга\n")

    result = await fetch_article_v4(
        article_80["href"], article_80["text"], article_hierarchy
    )

    if result and result["text_length"] > 300:
        print(f"\n{'=' * 80}")
        print("🎉 УСПЕХ! Статья загружена со ВСЕМИ метаданными")
        print(f"{'=' * 80}\n")
        print(f"📌 Заголовок: {result['title']}")
        print(f"📝 Длина: {result['text_length']} символов")
        print(f"🔖 Статус: {result['status']}")
        print(f"🗂️  Часть: {result['part'] or 'не найдено'}")
        print(f"🗂️  Раздел: {result['section'] or 'не найдено'}")
        print(f"🗂️  Глава: {result['chapter'] or 'не найдено'}")
        print(f"🔗 Источник: {result['source']}")
        print(f"🌐 URL: {result['source_url']}")
        print(f"📅 Загружено: {result['fetched_at']}")
        print("\n📄 Первые 500 символов:\n")
        print("=" * 80)
        print(result["text"][:500])
        print("=" * 80)

        # Сохраняем
        article_80_v4_final = result
    elif result:
        print(
            f"\n⚠️ Статья загружена, но текст короткий: {result['text_length']} символов"
        )
    else:
        print("❌ Ошибка загрузки")

🚀 Тест v4: с извлечением иерархии из маппинга

⏳ Загружаем: Статья 80. Расторжение трудового договора по иници...
⏳ Загружаем: Статья 80. Расторжение трудового договора по иници...
✓ Статья загружена: 3208 символов, статус=active
  Иерархия: Часть III → Раздел III. Трудовой договор → Глава 13. Прекращение трудового договора

🎉 УСПЕХ! Статья загружена со ВСЕМИ метаданными

📌 Заголовок: ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)
📝 Длина: 3208 символов
🔖 Статус: active
🗂️  Часть: Часть III
🗂️  Раздел: Раздел III. Трудовой договор
🗂️  Глава: Глава 13. Прекращение трудового договора
🔗 Источник: ConsultantPlus
🌐 URL: https://www.consultant.ru/document/cons_doc_LAW_34683/aed7d03df679e3376974dadd131b899dc6966650/
📅 Загружено: 2025-11-24T13:28:42.405752

📄 Первые 500 символов:

ТК РФ Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)

Путеводители по кадровым вопросам и трудовым спорам. Вопросы приме

### Тест на статье 7 (утратила силу)

Проверим, что функция правильно определяет статус `abolished`

In [24]:
# Найдем статью 7 (она утратила силу)
article_7 = None
for item in article_links:
    if (
        "Статья 7" in item["text"]
        and "Статья 70" not in item["text"]
        and "Статья 77" not in item["text"]
    ):
        article_7 = item
        print(f"✅ Найдена: {article_7['text']}")
        break

if article_7:
    print("\n🔍 Тестируем определение статуса 'abolished'\n")
    result = await fetch_article_v4(
        article_7["href"], article_7["text"], article_hierarchy
    )

    if result:
        print(f"\n{'=' * 80}")
        status_check = "✅" if result["status"] == "abolished" else "❌"
        print(f"🔖 Статус: {result['status']} {status_check}")
        print(f"📌 Заголовок: {result['title']}")
        print(f"📝 Длина: {result['text_length']} символов")
        print("\n📄 Текст:\n")
        print("=" * 80)
        print(result["text"][:400])
        print("=" * 80)
    else:
        print("❌ Ошибка загрузки")
else:
    print("❌ Статья 7 не найдена в списке")

✅ Найдена: Статья 7. Утратила силу

🔍 Тестируем определение статуса 'abolished'

⏳ Загружаем: Статья 7. Утратила силу...
⏳ Загружаем: Статья 7. Утратила силу...
✓ Статья загружена: 133 символов, статус=abolished
  Иерархия: Часть I → Раздел I. Общие положения → Глава 1. Основные начала трудового законодательства

🔖 Статус: abolished ✅
📌 Заголовок: ТК РФ Статья 7. Утратила силу
📝 Длина: 133 символов

📄 Текст:

ТК РФ Статья 7. Утратила силу

Статья 7. Утратила силу. - Федеральный закон от 30.06.2006 N 90-ФЗ.

(см. текст в предыдущей редакции)
✓ Статья загружена: 133 символов, статус=abolished
  Иерархия: Часть I → Раздел I. Общие положения → Глава 1. Основные начала трудового законодательства

🔖 Статус: abolished ✅
📌 Заголовок: ТК РФ Статья 7. Утратила силу
📝 Длина: 133 символов

📄 Текст:

ТК РФ Статья 7. Утратила силу

Статья 7. Утратила силу. - Федеральный закон от 30.06.2006 N 90-ФЗ.

(см. текст в предыдущей редакции)


### Обновленная функция parse_all_articles_v4

Используем fetch_article_v4 для парсинга всех статей с полными метаданными

In [25]:
from tqdm.asyncio import tqdm
import json
from pathlib import Path


async def parse_all_articles_v4(article_links, hierarchy_map, start_from=0):
    """
    Парсит все статьи с прогресс-баром (v4 - с полными метаданными)

    Args:
        article_links: список ссылок на статьи
        hierarchy_map: маппинг номер статьи → иерархия
        start_from: с какой статьи начать (для продолжения после ошибки)

    Returns:
        tuple: (список успешно спарсенных статей, список ошибок)
    """
    all_articles = []
    failed_articles = []

    print(f"🚀 Начинаем парсинг {len(article_links)} статей (v4 - с метаданными)...")
    print(f"⏱️  Примерное время: ~{len(article_links) * 30 // 60} минут\n")

    # Используем tqdm для прогресс-бара
    for i, article_link in enumerate(
        tqdm(
            article_links[start_from:],
            desc="Парсинг статей v4",
            initial=start_from,
            total=len(article_links),
        )
    ):
        actual_index = start_from + i

        try:
            result = await fetch_article_v4(
                article_link["href"], article_link["text"], hierarchy_map
            )

            if result and result["text_length"] > 50:  # Минимум 50 символов
                all_articles.append(result)
            elif result:
                # Статья загрузилась, но текст очень короткий
                failed_articles.append(
                    {
                        "index": actual_index,
                        "title": article_link["text"],
                        "url": article_link["href"],
                        "reason": f"Текст короткий: {result['text_length']} символов",
                    }
                )
            else:
                # Ошибка загрузки
                failed_articles.append(
                    {
                        "index": actual_index,
                        "title": article_link["text"],
                        "url": article_link["href"],
                        "reason": "Ошибка загрузки",
                    }
                )

        except Exception as e:
            failed_articles.append(
                {
                    "index": actual_index,
                    "title": article_link["text"],
                    "url": article_link["href"],
                    "reason": str(e),
                }
            )

        # Сохраняем промежуточные результаты каждые 10 статей
        if (actual_index + 1) % 10 == 0:
            save_checkpoint_v4(all_articles, failed_articles, actual_index + 1)

    return all_articles, failed_articles


def save_checkpoint_v4(articles, failed, count):
    """Сохраняет промежуточные результаты (v4 с метаданными)"""
    checkpoint_dir = Path("../data/checkpoints")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_file = checkpoint_dir / f"articles_v4_checkpoint_{count}.json"
    checkpoint_file.write_text(
        json.dumps(
            {"count": count, "articles": articles, "failed": failed},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(f"💾 Чекпоинт сохранен: {checkpoint_file.name}")


print("✅ Функция parse_all_articles_v4 готова")

✅ Функция parse_all_articles_v4 готова


### Запуск парсинга

**Внимание**: Это займет ~4.5 часа для 537 статей (по 30 секунд на статью)

Промежуточные результаты сохраняются каждые 10 статей в `data/checkpoints/`

In [29]:
# Запускаем парсинг всех статей
# Чтобы начать с определенной статьи (после ошибки), используйте start_from=N
all_articles, failed_articles = await parse_all_articles_v4(
    article_links[:8], article_hierarchy, start_from=0
)

print(f"\n{'=' * 80}")
print("✅ Парсинг завершен!")
print(f"{'=' * 80}\n")
print(f"📊 Успешно спарсено: {len(all_articles)} статей")
print(f"❌ Ошибок: {len(failed_articles)} статей")

if failed_articles:
    print("\n⚠️  Статьи с ошибками:")
    for item in failed_articles[:10]:
        print(f"  - {item['title']}: {item['reason']}")

🚀 Начинаем парсинг 8 статей (v4 - с метаданными)...
⏱️  Примерное время: ~4 минут



Парсинг статей v4:   0%|          | 0/8 [00:00<?, ?it/s]

⏳ Загружаем: Статья 1. Цели и задачи трудового законодательства...


Парсинг статей v4:  12%|█▎        | 1/8 [00:04<00:28,  4.10s/it]

✓ Статья загружена: 1740 символов, статус=active
  Иерархия: Часть I → Раздел I. Общие положения → Глава 1. Основные начала трудового законодательства
⏳ Загружаем: Статья 2. Основные принципы правового регулировани...


...

✓ Статья загружена: 4565 символов, статус=active
  Иерархия: Часть I → Раздел I. Общие положения → Глава 1. Основные начала трудового законодательства
⏳ Загружаем: Статья 7. Утратила силу...


Парсинг статей v4:  88%|████████▊ | 7/8 [00:28<00:04,  4.03s/it]

✓ Статья загружена: 133 символов, статус=abolished
  Иерархия: Часть I → Раздел I. Общие положения → Глава 1. Основные начала трудового законодательства
⏳ Загружаем: Статья 8. Локальные нормативные акты, содержащие н...


Парсинг статей v4: 100%|██████████| 8/8 [00:32<00:00,  4.05s/it]

✓ Статья загружена: 1926 символов, статус=active
  Иерархия: Часть I → Раздел I. Общие положения → Глава 1. Основные начала трудового законодательства

✅ Парсинг завершен!

📊 Успешно спарсено: 8 статей
❌ Ошибок: 0 статей


## Шаг 6: Сохранение результатов в JSON

Сохраняем все статьи в финальный JSON файл

In [30]:
# Подготовка финального JSON
output_data = {
    "metadata": {
        "source": "ConsultantPlus",
        "url": "https://www.consultant.ru/document/cons_doc_LAW_34683/",
        "document": "Трудовой кодекс РФ",
        "parsed_at": datetime.now().isoformat(),
        "total_articles": len(all_articles),
        "failed_articles": len(failed_articles),
    },
    "articles": all_articles,
    "failed": failed_articles,
}

# Сохраняем в файл
output_file = Path("../data/tk_rf_articles.json")
output_file.parent.mkdir(parents=True, exist_ok=True)
output_file.write_text(
    json.dumps(output_data, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"✅ Данные сохранены в: {output_file}")
print(f"📊 Размер файла: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print("\n📝 Структура данных:")
print("  - metadata: информация о парсинге")
print(f"  - articles: {len(all_articles)} статей")
print(f"  - failed: {len(failed_articles)} ошибок")

✅ Данные сохранены в: ../data/tk_rf_articles.json
📊 Размер файла: 0.04 MB

📝 Структура данных:
  - metadata: информация о парсинге
  - articles: 8 статей
  - failed: 0 ошибок
